# Least-to-Most Prompting

## Lernziel

Du vergleichst zwei Wege für dieselbe komplexe Aufgabe:

1. **Direkt:** Report und Zielfrage gehen in einen Modellaufruf.
2. **Least-to-Most:** Das Modell zerlegt die Aufgabe, beantwortet die Teilfragen nacheinander und erstellt daraus die Endantwort.

Du erstellst nur die Prompts. Ausführung, Verkettung und Bewertung sind vorgegeben.


## 0 · Setup

Führe die nächsten beiden Zellen aus. Das in `helfer.py` konfigurierte Modell analysiert einen Incident-Report.


In [ ]:
import re
import sys
from pathlib import Path

try:
    import openai
except ImportError:
    %pip install -q openai

for kandidat in [Path.cwd(), *Path.cwd().parents, Path("/content"),
                  Path("/content/01_prompt-engineering")]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

from helfer import BASIS_URL, MODELL, frage_llm, lade_daten, zeige

IR = lade_daten("incident_report")
REPORT = IR["report"]
ZIELFRAGE = IR["frage"]
PRUEFPUNKTE = lade_daten("06_pruefpunkte")["punkte"]

print(f"Server: {BASIS_URL}")
print(f"Modell: {MODELL}")
print(f"Incident: {IR['id']} — {IR['titel']}")


In [ ]:
print("Testantwort:", frage_llm("Reply with exactly: ready"))


## 1 · Die Aufgabe des Modells

Der Report beschreibt einen möglichen Ransomware-Angriff. Das Modell soll entscheiden:

> Welche Systeme müssen zuerst vom Netz, und in welcher Reihenfolge soll das Response-Team in den nächsten vier Stunden handeln?

Eine gute Antwort muss mehrere Hinweise aus dem Report verbinden: betroffene Hosts, ein missbrauchtes Dienstkonto, den Credential-Vault, eine bekannte CVE und den verschlüsselten Fileserver.

Least-to-Most soll diese zusammengesetzte Aufgabe zunächst in kleinere Fragen zerlegen. Danach wird jede Teilfrage mit den bisherigen Antworten im Kontext gelöst.


In [ ]:
zeige(REPORT, titel="Incident-Report")
zeige(ZIELFRAGE, titel="Zielfrage")

print("Prüfpunkte für den späteren Vergleich:")
for nummer, punkt in enumerate(IR["pruefpunkte"], start=1):
    print(f"  {nummer}. {punkt}")


In [ ]:
def deckt_ab(antwort):
    saetze = [satz.strip() for satz in re.split(r"(?<=[.!?])\s+|\n+", antwort.lower())
              if satz.strip()]
    getroffen = []
    for eintrag in PRUEFPUNKTE:
        treffer = any(
            all(any(signal in satz for signal in gruppe) for gruppe in eintrag["signale"])
            for satz in saetze
        )
        if treffer:
            getroffen.append(eintrag["punkt"])
    return {
        "quote": len(getroffen) / len(PRUEFPUNKTE),
        "getroffen": getroffen,
        "fehlend": [p["punkt"] for p in PRUEFPUNKTE if p["punkt"] not in getroffen],
    }


def hole_teilfragen(text):
    """Liest nummerierte oder mit Aufzählungszeichen markierte Fragen aus einer Antwort."""
    marke = re.compile(r"^\s*(?:\d+[.)]|[-*•])\s+")
    fragen = []
    for zeile in text.splitlines():
        if marke.match(zeile):
            frage = marke.sub("", zeile).strip().strip("*").strip()
            if frage.endswith("?"):
                fragen.append(frage)
    return fragen


print("Bewertung und Hilfsfunktionen sind bereit.")


## Challenge 1 · Direkte Baseline

Implementiere `baue_direkt_prompt(report, zielfrage)`.

Der Prompt soll enthalten:

- die Rolle eines Incident-Response-Leads,
- den vollständigen Report,
- die Zielfrage,
- die Aufforderung zu einem nummerierten, begründeten Maßnahmenplan mit konkreten Namen.

Dieser Prompt nutzt keine Zerlegung. Er dient später als Vergleich.


In [ ]:
def baue_direkt_prompt(report, zielfrage):
    # TODO: Erstelle den direkten Prompt aus Report und Zielfrage.
    pass


In [ ]:
probe_direkt = baue_direkt_prompt("EINDEUTIGER-REPORT", "EINDEUTIGE-FRAGE?")
assert isinstance(probe_direkt, str), "Der Prompt muss ein String sein."
assert "EINDEUTIGER-REPORT" in probe_direkt, "Der Report fehlt."
assert "EINDEUTIGE-FRAGE?" in probe_direkt, "Die Zielfrage fehlt."
assert "order" in probe_direkt.lower() or "numbered" in probe_direkt.lower(), \
    "Fordere eine klare Reihenfolge an."
print("✅ Direkter Prompt ist vollständig.\n")
print(probe_direkt)


In [ ]:
antwort_direkt = frage_llm(baue_direkt_prompt(REPORT, ZIELFRAGE), max_tokens=700)
zeige(antwort_direkt, titel="Direkte Antwort")


## Challenge 2 · Aufgabe zerlegen

Implementiere `baue_zerlege_prompt(report, zielfrage)`.

Das Modell soll die Zielfrage **noch nicht beantworten**, sondern genau vier einfachere Teilfragen formulieren. Der Prompt soll festlegen:

- sinnvolle Reihenfolge,
- nur eine nummerierte Liste,
- eine Frage pro Zeile,
- jede Zeile endet mit `?`.

Das klare Ausgabeformat ermöglicht der vorgegebenen Funktion `hole_teilfragen(...)`, die Fragen zuverlässig einzulesen.


In [ ]:
def baue_zerlege_prompt(report, zielfrage):
    # TODO: Erstelle den Prompt, der die Aufgabe in genau vier Teilfragen zerlegt.
    pass


In [ ]:
probe_zerlegung = baue_zerlege_prompt("EINDEUTIGER-REPORT", "EINDEUTIGE-FRAGE?")
assert isinstance(probe_zerlegung, str), "Der Prompt muss ein String sein."
assert "EINDEUTIGER-REPORT" in probe_zerlegung, "Der Report fehlt."
assert "EINDEUTIGE-FRAGE?" in probe_zerlegung, "Die Zielfrage fehlt."
klein = probe_zerlegung.lower()
assert "do not answer" in klein, "Die Hauptfrage soll noch nicht beantwortet werden."
assert "four" in klein or "4" in klein, "Fordere genau vier Teilfragen an."
assert "question mark" in klein or "?" in probe_zerlegung, "Fordere Fragezeichen an."
print("✅ Zerlege-Prompt ist vollständig.\n")
print(probe_zerlegung)


In [ ]:
roh_zerlegung = frage_llm(baue_zerlege_prompt(REPORT, ZIELFRAGE), max_tokens=350)
zeige(roh_zerlegung, titel="Antwort auf den Zerlege-Prompt")

TEILFRAGEN = hole_teilfragen(roh_zerlegung)
if not TEILFRAGEN:
    raise ValueError("Keine Teilfragen erkannt. Prüfe den Zerlege-Prompt und die Ausgabe.")

print(f"{len(TEILFRAGEN)} Teilfragen erkannt:")
for nummer, frage in enumerate(TEILFRAGEN, start=1):
    print(f"  {nummer}. {frage}")


## Challenge 3 · Teilfragen verketten

Implementiere `baue_teilfragen_prompt(report, teilfrage, bisherige_paare)`.

Für jede Teilfrage soll der Prompt enthalten:

- den vollständigen Report,
- die aktuelle Teilfrage,
- alle bisherigen Frage-Antwort-Paare, sofern bereits welche existieren,
- die Vorgabe, nur diese Teilfrage kurz und konkret zu beantworten.

Genau diese wachsende Liste bisheriger Antworten macht aus mehreren unabhängigen Fragen eine Least-to-Most-Kette.


In [ ]:
def baue_teilfragen_prompt(report, teilfrage, bisherige_paare):
    # TODO: Erstelle den Prompt für eine Teilfrage.
    # Ab der zweiten Teilfrage müssen alle bisherigen Frage-Antwort-Paare enthalten sein.
    pass


In [ ]:
probe_paare = [("Frage 1?", "Antwort 1"), ("Frage 2?", "Antwort 2")]
probe_teilfrage = baue_teilfragen_prompt("EINDEUTIGER-REPORT", "Frage 3?", probe_paare)
assert isinstance(probe_teilfrage, str), "Der Prompt muss ein String sein."
assert "EINDEUTIGER-REPORT" in probe_teilfrage, "Der Report fehlt."
assert "Frage 3?" in probe_teilfrage, "Die aktuelle Teilfrage fehlt."
for frage, antwort in probe_paare:
    assert frage in probe_teilfrage and antwort in probe_teilfrage, \
        "Alle bisherigen Frage-Antwort-Paare müssen enthalten sein."

probe_erste = baue_teilfragen_prompt("REPORT", "Erste Frage?", [])
assert "Antwort 1" not in probe_erste, "Vor der ersten Frage gibt es keine frühere Antwort."
print("✅ Prompt für die verketteten Teilfragen ist vollständig.\n")
print(probe_teilfrage)


In [ ]:
PAARE = []
for nummer, teilfrage in enumerate(TEILFRAGEN, start=1):
    prompt = baue_teilfragen_prompt(REPORT, teilfrage, PAARE)
    antwort = frage_llm(prompt, max_tokens=300)
    PAARE.append((teilfrage, antwort))
    zeige(antwort, titel=f"Teilfrage {nummer}: {teilfrage}")


## Challenge 4 · Endantwort erstellen

Implementiere `baue_endprompt(report, zielfrage, paare)`.

Der Prompt führt alles zusammen:

- den Report,
- alle Teilfragen mit ihren Antworten,
- die ursprüngliche Zielfrage,
- die Aufforderung zu einem nummerierten Maßnahmenplan in begründeter Reihenfolge.

Die Teilantworten sind Kontext, nicht die fertige Ausgabe. Das Modell soll daraus eine zusammenhängende Endantwort erzeugen.


In [ ]:
def baue_endprompt(report, zielfrage, paare):
    # TODO: Erstelle den Endprompt aus Report, Teilantworten und Zielfrage.
    pass


In [ ]:
probe_endprompt = baue_endprompt(
    "EINDEUTIGER-REPORT", "EINDEUTIGE-FRAGE?", [("Teilfrage?", "Teilantwort")]
)
assert isinstance(probe_endprompt, str), "Der Prompt muss ein String sein."
for text in ["EINDEUTIGER-REPORT", "EINDEUTIGE-FRAGE?", "Teilfrage?", "Teilantwort"]:
    assert text in probe_endprompt, f"{text!r} fehlt im Endprompt."
assert "order" in probe_endprompt.lower() or "numbered" in probe_endprompt.lower(), \
    "Fordere eine klare Reihenfolge an."
print("✅ Endprompt ist vollständig.\n")
print(probe_endprompt)


## 2 · Least-to-Most ausführen und vergleichen

Der letzte Modellaufruf erzeugt nun die Endantwort aus den Teilantworten. Danach werden direkte Antwort und Least-to-Most-Antwort mit denselben Prüfpunkten verglichen.


In [ ]:
antwort_l2m = frage_llm(baue_endprompt(REPORT, ZIELFRAGE, PAARE), max_tokens=700)
zeige(antwort_l2m, titel="Least-to-Most-Endantwort")

ERGEBNISSE = {
    "Direkt": deckt_ab(antwort_direkt),
    "Least-to-Most": deckt_ab(antwort_l2m),
}

print(f"{'Prüfpunkt':<62} {'Direkt':>8} {'L-to-M':>8}")
print("─" * 80)
for eintrag in PRUEFPUNKTE:
    punkt = eintrag["punkt"]
    direkt = "✔" if punkt in ERGEBNISSE["Direkt"]["getroffen"] else "✘"
    l2m = "✔" if punkt in ERGEBNISSE["Least-to-Most"]["getroffen"] else "✘"
    print(f"{punkt:<62} {direkt:>8} {l2m:>8}")
print("─" * 80)
print(f"{'Abdeckung':<62} {ERGEBNISSE['Direkt']['quote']:>8.0%} "
      f"{ERGEBNISSE['Least-to-Most']['quote']:>8.0%}")


## 3 · Offene Diskussion

Betrachtet die Antworten, Teilantworten und die Tabelle und besprecht:

1. Welche Variante deckt bei eurem Modell mehr Prüfpunkte ab?
2. Welche Information wurde durch die Zerlegung sichtbar — und welche ging eventuell verloren?
3. Haben die frühen Teilantworten die späteren Antworten sinnvoll unterstützt oder in eine falsche Richtung gelenkt?
4. Ist die Least-to-Most-Endantwort nur vollständiger oder auch sachlich besser priorisiert?
5. Rechtfertigt der Unterschied die zusätzlichen Modellaufrufe?

Es gibt kein vorgegebenes Gewinnergebnis. Least-to-Most kann besser, gleich gut oder schlechter abschneiden. Entscheidend ist, dass ihr nachvollziehen könnt, **wo** ein Unterschied entstanden ist: bei der Zerlegung, in einer Teilantwort oder beim Zusammenführen.
